# Outlier-Aware Classifier

An experiment for training a classifier on a small dataset (with the in-domain labelled set being even smaller). The goal is maximizing in-domain accuracy, while pushing unlabeled out-domain images toward random-ish predictions.

## Imports

In [ ]:
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence
from contextlib import nullcontext
import random
import time
from datetime import datetime

import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch import nn
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms, models
from torchvision.transforms import functional as TF
from torchvision.transforms import v2 as T
from torchvision.transforms.functional import InterpolationMode

torch.set_float32_matmul_precision("high")

## Configuration

`FULL_RUN = False` checks that the notebook runs. `FULL_RUN = True` uses the longer run. The dataset paths assume the archive has already been unzipped into the runtime working directory.

In [ ]:
@dataclass
class Config:
    tensor_size: int
    batch_size: int
    stage0_batch_size: int
    stage0_epochs: int
    stage1_epochs: int
    stage2_epochs: int
    stage0_lr: float
    stage1_lr: float
    stage2_lr: float
    entropy_lambda: float
    snapshot_count: int
    seed: int
    log_dir: Path


FULL_RUN = True

CFG = Config(
    tensor_size=320,
    batch_size=32,
    stage0_batch_size=64,
    stage0_epochs=64 if FULL_RUN else 4,
    stage1_epochs=128 if FULL_RUN else 4,
    stage2_epochs=128 if FULL_RUN else 6,
    stage0_lr=1e-3,
    stage1_lr=1e-3,
    stage2_lr=1e-3,
    entropy_lambda=1.0,
    snapshot_count=5,
    seed=42,
    log_dir=Path("runs/colab_snapshot_demo"),
)

DATA_ROOT = Path(".")
PATHS = {
    "in_train": DATA_ROOT / "in-domain-train",
    "out_train": DATA_ROOT / "out-domain-train",
    "in_eval": DATA_ROOT / "in-domain-eval",
    "out_eval": DATA_ROOT / "out-domain-eval",
}

## Optional Colab Data Prep

These lines are left commented because the notebook expects the folders to already be unzipped in the runtime working directory.


In [ ]:
# uncomment for mounting drive
from google.colab import drive
drive.mount('/content/drive')

# uncomment if the archive needs to be copied and unzipped
!cp drive/MyDrive/A4data.zip .
!unzip -q A4data.zip


## Utilities

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def count_jpgs(path: Path) -> int:
    return sum(1 for p in path.rglob("*.jpg") if p.is_file())


def print_rows(rows: Sequence[Dict[str, object]]) -> None:
    if not rows:
        print("No rows")
        return
    keys = list(rows[0].keys())
    widths = {k: max(len(str(k)), *(len(str(row.get(k, ""))) for row in rows)) for k in keys}
    print("  ".join(str(k).ljust(widths[k]) for k in keys))
    print("  ".join("-" * widths[k] for k in keys))
    for row in rows:
        print("  ".join(str(row.get(k, "")).ljust(widths[k]) for k in keys))


set_seed(CFG.seed)
print(get_device())
print_rows([
    {"split": name, "path": str(path), "jpgs": count_jpgs(path)}
    for name, path in PATHS.items()
])

## Image Loading

Images are loaded once into memory after letterboxing. Training keeps uint8 tensors and applies augmentation later on-device. Evaluation stores normalized float tensors. This significantly reduces IO overhead.

In [ ]:
def letterbox_to_square(img: Image.Image, size: int) -> Image.Image:
    w, h = img.size
    if w == h or w == 0 or h == 0:
        return img.resize((size, size), Image.BICUBIC)
    scale = size / float(max(w, h))
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    img = img.resize((new_w, new_h), Image.BICUBIC)
    pad_w = size - new_w
    pad_h = size - new_h
    padding = (pad_w // 2, pad_h // 2, pad_w - pad_w // 2, pad_h - pad_h // 2)
    return TF.pad(img, padding, padding_mode="edge")


class InMemoryClassImageFolder(Dataset):
    def __init__(self, root: Path, train: bool, tensor_size: int):
        base = datasets.ImageFolder(str(root), is_valid_file=lambda p: p.lower().endswith(".jpg"))
        self.samples = base.samples
        self.targets = list(getattr(base, "targets", [s[1] for s in base.samples]))
        self.class_to_idx = base.class_to_idx
        self.classes = base.classes
        self.train = train
        self.images: List[torch.Tensor] = []
        for path, _ in self.samples:
            with Image.open(path) as img:
                img = letterbox_to_square(img.convert("RGB"), tensor_size)
                tensor = TF.pil_to_tensor(img) if train else TF.to_tensor(img)
                self.images.append(tensor)
        self.normalize = None if train else transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        x = self.images[idx]
        y = self.targets[idx]
        if self.normalize is not None:
            x = self.normalize(x)
        return x, y


class InMemoryOutDomainDataset(Dataset):
    def __init__(self, root: Path, train: bool, tensor_size: int):
        self.paths = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".jpg"]
        self.train = train
        self.images: List[torch.Tensor] = []
        for path in self.paths:
            with Image.open(path) as img:
                img = letterbox_to_square(img.convert("RGB"), tensor_size)
                tensor = TF.pil_to_tensor(img) if train else TF.to_tensor(img)
                self.images.append(tensor)
        self.normalize = None if train else transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        x = self.images[idx]
        if self.normalize is not None:
            x = self.normalize(x)
        return x


class ContrastiveDataset(Dataset):
    def __init__(self, in_dataset: InMemoryClassImageFolder, out_dataset: InMemoryOutDomainDataset):
        self.images = list(in_dataset.images) + list(out_dataset.images)

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        return self.images[idx]


def make_loader(dataset: Dataset, train: bool, batch_size: int) -> DataLoader:
    return DataLoader(dataset, batch_size=batch_size, shuffle=train, drop_last=train, pin_memory=True)


def infinite_loader(loader: DataLoader):
    while True:
        for batch in loader:
            yield batch

## Augmentation and Mixed Precision

The same augmentation idea is used for supervised and entropy fine-tuning. SimCLR uses a stronger two-view transform.

In [ ]:
def amp_context(device: torch.device):
    if device.type == "cuda":
        return torch.amp.autocast(device_type="cuda")
    return nullcontext()


class DummyScaler:
    def scale(self, loss):
        return loss

    def step(self, optimizer):
        optimizer.step()

    def update(self):
        pass


def make_scaler(device: torch.device):
    if device.type == "cuda":
        return torch.amp.GradScaler("cuda")
    return DummyScaler()


def supervised_transform(device: torch.device, tensor_size: int) -> nn.Module:
    blur_kernel = max(1, tensor_size // 10)
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    transform = T.Compose([
        T.RandomAffine(degrees=20.0, scale=(0.9, 1.1), shear=(-10.0, 10.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15),
        T.RandomGrayscale(p=0.2),
        T.RandomApply([T.GaussianBlur(kernel_size=blur_kernel, sigma=(0.1, 2.0))], p=0.5),
        T.RandomErasing(p=0.2, scale=(0.02, 0.2), value=0.0),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])
    return transform.to(device)


def simclr_transform(device: torch.device, tensor_size: int) -> nn.Module:
    blur_kernel = max(1, tensor_size // 10)
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    transform = T.Compose([
        T.RandomResizedCrop(size=(tensor_size, tensor_size), scale=(0.4, 1.0), interpolation=InterpolationMode.BILINEAR, antialias=True),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomApply([T.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.RandomApply([T.GaussianBlur(kernel_size=blur_kernel, sigma=(0.1, 2.0))], p=0.5),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])
    return transform.to(device)

## Model

We use ResNet-18 to take advantage of its simple design. The classifier head uses cosine similarity so class decisions are made by angular similarity in feature space.

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 512, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(inplace=True), nn.Linear(hidden_dim, out_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CosineClassifier(nn.Module):
    def __init__(self, in_features: int, num_classes: int, scale: float = 30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        self.logit_scale = nn.Parameter(torch.tensor(float(scale)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = F.normalize(x, dim=1)
        w_norm = F.normalize(self.weight, dim=1)
        return F.linear(x_norm, w_norm) * self.logit_scale


def build_model(num_classes: int, init_state: Dict[str, torch.Tensor] | None = None) -> nn.Module:
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = CosineClassifier(in_features, num_classes)
    if init_state is not None:
        model.load_state_dict(init_state, strict=False)
    return model


def set_bn_eval(model: nn.Module) -> None:
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def freeze_except_layer4_and_fc(model: nn.Module) -> List[nn.Parameter]:
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    set_bn_eval(model)
    return [p for p in model.parameters() if p.requires_grad]

## Evaluation

The table compares in-domain accuracy, out-domain accuracy, the gap between them, and entropy on each split.

In [ ]:
def align_dataset_to_class_map(dataset: InMemoryClassImageFolder, class_map: Dict[str, int]) -> None:
    remap = {dataset.class_to_idx[name]: class_map[name] for name in dataset.classes}
    dataset.samples = [(sample_path, remap[target]) for sample_path, target in dataset.samples]
    dataset.targets = [remap[target] for target in dataset.targets]
    dataset.class_to_idx = class_map.copy()


def prepare_eval_loader(path: Path, class_map: Dict[str, int], cfg: Config) -> DataLoader:
    dataset = InMemoryClassImageFolder(path, train=False, tensor_size=cfg.tensor_size)
    align_dataset_to_class_map(dataset, class_map)
    return make_loader(dataset, train=False, batch_size=256)


def model_from_artifact(artifact: Dict[str, object], device: torch.device) -> nn.Module:
    class_map = artifact["classes"]
    model = build_model(len(class_map))
    model.load_state_dict(artifact["state"])
    model.to(device)
    model.eval()
    return model


def evaluate_model(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    correct = 0
    total = 0
    entropy_sum = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
            correct += int((preds == labels).sum().item())
            total += int(labels.numel())
            entropy_sum += float(entropy.sum().item())
    return {"accuracy": correct / max(1, total), "entropy": entropy_sum / max(1, total), "n": float(total)}

## Snapshot Voting

The last Stage 2 checkpoints are treated as a small ensemble. For each image, the notebook averages probabilities from the selected snapshots and predicts from the averaged distribution.

In [ ]:
def clone_artifact(model: nn.Module, class_map: Dict[str, int], epoch: int, metrics: Dict[str, float]) -> Dict[str, object]:
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return {"state": state, "classes": class_map.copy(), "epoch": epoch, "metrics": metrics.copy()}


def evaluate_snapshots(snapshots: Sequence[Dict[str, object]], path: Path, cfg: Config) -> Dict[str, float]:
    if len(snapshots) == 0:
        raise ValueError("No snapshots provided")
    device = get_device()
    class_map = snapshots[0]["classes"]
    loader = prepare_eval_loader(path, class_map, cfg)
    models_loaded = [model_from_artifact(snapshot, device) for snapshot in snapshots]
    correct = 0
    total = 0
    entropy_sum = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            prob_sum = None
            for model in models_loaded:
                probs = torch.softmax(model(images), dim=1)
                prob_sum = probs if prob_sum is None else prob_sum + probs
            avg_probs = prob_sum / len(models_loaded)
            preds = avg_probs.argmax(dim=1)
            entropy = -torch.sum(avg_probs * torch.log(avg_probs + 1e-8), dim=1)
            correct += int((preds == labels).sum().item())
            total += int(labels.numel())
            entropy_sum += float(entropy.sum().item())
    for model in models_loaded:
        model.cpu()
    return {"accuracy": correct / max(1, total), "entropy": entropy_sum / max(1, total), "n": float(total)}

## Stage 0

This stage uses SimCLR-style contrastive pretraining over both in-domain and out-domain training images. This takes advantage of unlabelled data, hoping that the distinct features on OOD items can be learned by the model.

In [ ]:
def contrastive_pretrain(in_dataset: InMemoryClassImageFolder, out_dataset: InMemoryOutDomainDataset, cfg: Config) -> Dict[str, torch.Tensor]:
    device = get_device()
    dataset = ContrastiveDataset(in_dataset, out_dataset)
    loader = DataLoader(dataset, batch_size=cfg.stage0_batch_size, shuffle=True, drop_last=True, pin_memory=True)
    encoder = models.resnet18(weights=None)
    feature_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()
    projector = ProjectionHead(feature_dim)
    encoder.to(device)
    projector.to(device)
    transform = simclr_transform(device, cfg.tensor_size)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(projector.parameters()), lr=cfg.stage0_lr, weight_decay=1e-4)
    scaler = make_scaler(device)
    for epoch in range(cfg.stage0_epochs):
        encoder.train()
        projector.train()
        transform.train()
        losses = []
        for images in loader:
            images = images.to(device, non_blocking=True)
            with amp_context(device):
                view1 = transform(images)
                view2 = transform(images)
                z1 = projector(encoder(view1))
                z2 = projector(encoder(view2))
                batch_size = z1.size(0)
                representations = torch.cat([z1, z2], dim=0)
                representations = F.normalize(representations, dim=1)
                similarity = representations @ representations.T
                similarity = similarity / 0.2
                mask = torch.eye(2 * batch_size, dtype=torch.bool, device=similarity.device)
                similarity = similarity.masked_fill(mask, -6e4)
                positives = torch.arange(batch_size, device=similarity.device)
                targets = torch.cat([positives + batch_size, positives], dim=0)
                loss = F.cross_entropy(similarity, targets)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.item()))
        print(datetime.now())
        print(f"[Stage0] Epoch {epoch + 1}/{cfg.stage0_epochs} nt_xent_loss={np.mean(losses):.4f}")
    state = encoder.cpu().state_dict()
    projector.cpu()
    return state

## Stage 1

This stage trains the classifier on labeled in-domain images.

In [ ]:
def train_in_domain_only(init_state: Dict[str, torch.Tensor], in_dataset: InMemoryClassImageFolder, cfg: Config) -> Dict[str, object]:
    device = get_device()
    loader = make_loader(in_dataset, train=True, batch_size=cfg.batch_size)
    model = build_model(len(in_dataset.classes), init_state).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.stage1_lr, weight_decay=1e-4)
    warmup_epochs = max(1, int(0.05 * cfg.stage1_epochs))
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=max(1, cfg.stage1_epochs - warmup_epochs), eta_min=0.01 * cfg.stage1_lr)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    criterion = nn.CrossEntropyLoss()
    transform = supervised_transform(device, cfg.tensor_size)
    scaler = make_scaler(device)
    for epoch in range(cfg.stage1_epochs):
        model.train()
        transform.train()
        correct = 0
        total = 0
        losses = []
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with amp_context(device):
                logits = model(transform(images))
                loss = criterion(logits, labels)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            preds = logits.argmax(dim=1)
            correct += int((preds == labels).sum().item())
            total += int(labels.numel())
            losses.append(float(loss.item()))
        scheduler.step()
        print(datetime.now())
        print(f"[Stage1] Epoch {epoch + 1}/{cfg.stage1_epochs} train_loss={np.mean(losses):.4f} train_acc={correct / max(1, total):.4f}")
    return {"state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "classes": in_dataset.class_to_idx.copy()}

## Stage 2

This stage trains only the last ResNet block and classifier head. The loss keeps supervised pressure on in-domain images and maximizes entropy on unlabeled out-domain images.

In [ ]:
def train_with_entropy_snapshots(base_artifact: Dict[str, object], in_dataset: InMemoryClassImageFolder, out_dataset: InMemoryOutDomainDataset, cfg: Config, paths: Dict[str, Path]) -> Dict[str, object]:
    device = get_device()
    class_map = base_artifact["classes"]
    align_dataset_to_class_map(in_dataset, class_map)
    in_batch_size = cfg.batch_size // 2
    out_batch_size = cfg.batch_size - in_batch_size
    in_loader = make_loader(in_dataset, train=True, batch_size=max(1, in_batch_size))
    out_loader = DataLoader(out_dataset, batch_size=max(1, out_batch_size), shuffle=True, drop_last=True, pin_memory=True)
    out_iter = infinite_loader(out_loader)
    eval_loaders = {
        "in": prepare_eval_loader(paths["in_eval"], class_map, cfg),
        "out": prepare_eval_loader(paths["out_eval"], class_map, cfg),
    }
    model = build_model(len(class_map))
    model.load_state_dict(base_artifact["state"])
    model.to(device)
    trainable_params = freeze_except_layer4_and_fc(model)
    optimizer = torch.optim.AdamW(trainable_params, lr=cfg.stage2_lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.stage2_epochs, eta_min=0.01 * cfg.stage2_lr)
    criterion = nn.CrossEntropyLoss()
    transform = supervised_transform(device, cfg.tensor_size)
    scaler = make_scaler(device)
    snapshots = deque(maxlen=cfg.snapshot_count)
    history = []
    writer = SummaryWriter(log_dir=str(cfg.log_dir / "stage2"))
    for epoch in range(cfg.stage2_epochs):
        started = time.time()
        model.train()
        transform.train()
        set_bn_eval(model)
        epoch_ce = 0.0
        epoch_loss = 0.0
        epoch_out_entropy = 0.0
        correct = 0
        total = 0
        batches = 0
        for images, labels in in_loader:
            out_images = next(out_iter)
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            out_images = out_images.to(device, non_blocking=True)
            with amp_context(device):
                logits = model(transform(images))
                ce_loss = criterion(logits, labels)
                out_logits = model(transform(out_images))
                out_probs = torch.softmax(out_logits, dim=1)
                out_entropy = -torch.sum(out_probs * torch.log(out_probs + 1e-8), dim=1).mean()
                loss = ce_loss - cfg.entropy_lambda * out_entropy
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            preds = logits.argmax(dim=1)
            correct += int((preds == labels).sum().item())
            total += int(labels.numel())
            epoch_ce += float(ce_loss.item())
            epoch_loss += float(loss.item())
            epoch_out_entropy += float(out_entropy.item())
            batches += 1
        in_metrics = evaluate_model(model, eval_loaders["in"], device)
        out_metrics = evaluate_model(model, eval_loaders["out"], device)
        metrics = {
            "epoch": float(epoch),
            "train_accuracy": correct / max(1, total),
            "loss_ce": epoch_ce / max(1, batches),
            "loss_total": epoch_loss / max(1, batches),
            "train_out_entropy": epoch_out_entropy / max(1, batches),
            "eval_in_accuracy": in_metrics["accuracy"],
            "eval_out_accuracy": out_metrics["accuracy"],
            "eval_in_entropy": in_metrics["entropy"],
            "eval_out_entropy": out_metrics["entropy"],
            "gap": in_metrics["accuracy"] - out_metrics["accuracy"],
            "seconds": time.time() - started,
        }
        history.append(metrics)
        snapshots.append(clone_artifact(model, class_map, epoch, metrics))
        writer.add_scalar("loss/ce", metrics["loss_ce"], epoch)
        writer.add_scalar("loss/total", metrics["loss_total"], epoch)
        writer.add_scalar("accuracy/train_in", metrics["train_accuracy"], epoch)
        writer.add_scalar("accuracy/eval_in", metrics["eval_in_accuracy"], epoch)
        writer.add_scalar("accuracy/eval_out", metrics["eval_out_accuracy"], epoch)
        writer.add_scalar("accuracy/gap", metrics["gap"], epoch)
        writer.add_scalar("entropy/train_out", metrics["train_out_entropy"], epoch)
        writer.add_scalar("entropy/eval_in", metrics["eval_in_entropy"], epoch)
        writer.add_scalar("entropy/eval_out", metrics["eval_out_entropy"], epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
        print(datetime.now())
        print(f"[Entropy] Epoch {epoch + 1}/{cfg.stage2_epochs} ce_loss={metrics['loss_ce']:.4f} total_loss={metrics['loss_total']:.4f} train_entropy_out={metrics['train_out_entropy']:.4f} train_acc={metrics['train_accuracy']:.4f} in_eval_acc={metrics['eval_in_accuracy']:.4f} out_eval_acc={metrics['eval_out_accuracy']:.4f} in_eval_H={metrics['eval_in_entropy']:.4f} out_eval_H={metrics['eval_out_entropy']:.4f} gap={metrics['gap']:.4f}")
        scheduler.step()
    writer.close()
    final_artifact = clone_artifact(model, class_map, cfg.stage2_epochs - 1, history[-1])
    model.cpu()
    return {"final": final_artifact, "snapshots": list(snapshots), "history": history}

## Load Data

This preloads the training images once before training starts.


In [ ]:
set_seed(CFG.seed)
train_in = InMemoryClassImageFolder(PATHS["in_train"], train=True, tensor_size=CFG.tensor_size)
train_out = InMemoryOutDomainDataset(PATHS["out_train"], train=True, tensor_size=CFG.tensor_size)


## Run Training

This runs the three stages after the training images are already loaded.


In [ ]:
stage0_state = contrastive_pretrain(train_in, train_out, CFG)
stage1_artifact = train_in_domain_only(stage0_state, train_in, CFG)
stage2_result = train_with_entropy_snapshots(stage1_artifact, train_in, train_out, CFG, PATHS)


## Compare Single Model and Snapshot Voting

`k=1` is just the last model. Larger values average the last `k` snapshots.

In [ ]:
def compare_snapshot_counts(stage2_result: Dict[str, object], cfg: Config, paths: Dict[str, Path], counts: Iterable[int]) -> List[Dict[str, object]]:
    snapshots = stage2_result["snapshots"]
    rows = []
    for k in counts:
        if k > len(snapshots):
            continue
        selected = snapshots[-k:]
        in_metrics = evaluate_snapshots(selected, paths["in_eval"], cfg)
        out_metrics = evaluate_snapshots(selected, paths["out_eval"], cfg)
        rows.append({
            "k": k,
            "in_acc": f"{in_metrics['accuracy']:.4f}",
            "out_acc": f"{out_metrics['accuracy']:.4f}",
            "gap": f"{in_metrics['accuracy'] - out_metrics['accuracy']:.4f}",
            "in_H": f"{in_metrics['entropy']:.4f}",
            "out_H": f"{out_metrics['entropy']:.4f}",
        })
    return rows


comparison = compare_snapshot_counts(stage2_result, CFG, PATHS, counts=[1, 3, 5, 10])
print_rows(comparison)

## TensorBoard

The scalar curves show whether entropy fine-tuning widens the gap or simply damages both domains.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs